# Lektion 05 - Agentisk RAG


## Opsætning

Denne notesbog demonstrerer Agentic RAG (Retrieval-Augmented Generation) mønstret ved brug af Microsoft Agent Framework.

**Forudsætninger:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — dit Azure AI Search service-endpoint
- `AZURE_SEARCH_API_KEY` — din Azure AI Search API-nøgle
- Azure OpenAI-udrulning konfigureret via miljøvariabler
- Azure CLI autentificeret (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hvad er Agentic RAG?

Traditionel RAG følger en fast pipeline: hent dokumenter, og generer derefter et svar. **Agentic RAG** går videre ved at give agenten autonomi til at beslutte **hvornår** og **hvordan** man skal hente information.

Med Agentic RAG kan agenten:
- **Beslutte**, om hentning er nødvendig, før der svares på et spørgsmål
- **Vælge**, hvilken datakilde eller værktøj der skal forespørges
- **Vurdere** hentede resultater og udføre opfølgende hentninger, hvis det første forsøg er utilstrækkeligt
- **Kombinere** information fra flere hentetrin til et sammenhængende svar

Dette gør agenten mere fleksibel og præcis sammenlignet med en statisk hent-og-generer pipeline.


## Oprettelse af et søgeværktøj

I Agentic RAG bliver eksterne datakilder indpakket som **værktøjer**, som agenten kan anvende efter behov. Dette gør det muligt for agenten at betragte hentning som blot en anden handling, den kan udføre, i stedet for et obligatorisk trin.

Nedenfor definerer vi en rejsevidensbase og eksponerer den som et værktøj, som agenten kan kalde for at slå destinationsinformation op.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Oprettelse af RAG-agenten

Nu opretter vi en agent, der er instrueret til **altid at hente information, før den svarer**. Agenten bruger værktøjet `search_travel_knowledge` til at forankre sine svar i vidensbasen i stedet for at stole på sine egne træningsdata.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativ hentning — Maker-Checker-mønsteret

En nøglefordel ved Agentic RAG er **iterativ hentning**. Agenten kan udføre flere søgerunder for at verificere, forfine eller udvide sine oprindelige fund — lignende en "maker-checker"-arbejdsgang:

1. **Maker-trin**: Agenten henter indledende information og udarbejder et svar.
2. **Checker-trin**: Agenten udfører yderligere hentninger for at bekræfte detaljer eller udfylde huller.

Nedenfor bliver agenten stillet et spørgsmål, der kræver sammenligning af flere destinationer, hvilket får den til at søge flere gange.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Resumé

I denne lektion lærte du, hvordan man bygger et **Agentic RAG**-system ved hjælp af Microsoft Agent Framework:

- **Agentic RAG** lader agenter autonomt afgøre, hvornår de skal hente information, hvilket gør hentningen dynamisk i stedet for fast.
- **Værktøjer som datakilder**: Eksterne vidensbaser (som Azure AI Search) pakkes ind som værktøjer, som agenten kan kalde.
- **Iterativ hentning**: Maker-checker-mønsteret gør det muligt for agenten at udføre flere hentningsrunder — søge, verificere og forfine — før den leverer et endeligt svar.

I produktion ville du erstatte den midlertidige `TRAVEL_KNOWLEDGE_BASE` med en rigtig Azure AI Search-indeks for at håndtere storskala rejsedokumenthentning.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
